# Comparación de modelos MLP vs EfficientNetB0

## 1. Configuración y carga de datos

In [ ]:
import os
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

IMG_SIZE = 128
BATCH_SIZE = 32

base_dir = '../data'
test_dir = os.path.join(base_dir, 'test')

# Generador para EfficientNet (con preprocess_input)
test_datagen_dl = ImageDataGenerator(preprocessing_function=preprocess_input)
test_generator_dl = test_datagen_dl.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Generador para MLP (sin preprocess, solo rescaling a 0-1)
test_datagen_mlp = ImageDataGenerator(rescale=1./255)
test_generator_mlp = test_datagen_mlp.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_names = list(test_generator_dl.class_indices.keys())
print('Clases:', class_names)

## 2. Cargar modelos entrenados

In [ ]:
mlp_path = '../models/mlp_best_model.keras'
dl_path = '../models/dl_best_model.keras'

mlp_model = tf.keras.models.load_model(mlp_path)
dl_model = tf.keras.models.load_model(dl_path)

print('Modelos cargados:')
print(' - MLP:', mlp_path)
print(' - EfficientNetB0:', dl_path)

## 3. Evaluación en conjunto de prueba

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

test_generator_mlp.reset()
test_generator_dl.reset()

X_test_mlp = []

for _ in range(int(np.ceil(test_generator_mlp.samples / BATCH_SIZE))):
    x_batch, _ = next(test_generator_mlp)
    X_test_mlp.append(x_batch)

X_test_mlp = np.vstack(X_test_mlp)

y_true = test_generator_dl.classes

mlp_pred = np.argmax(
    mlp_model.predict(X_test_mlp, batch_size=64, verbose=1),
    axis=1
)

test_generator_dl.reset()

dl_pred = np.argmax(
    dl_model.predict(test_generator_dl, verbose=1),
    axis=1
)

mlp_metrics = {
    'accuracy': accuracy_score(y_true, mlp_pred),
    'precision': precision_score(y_true, mlp_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_true, mlp_pred, average='weighted', zero_division=0),
    'f1': f1_score(y_true, mlp_pred, average='weighted', zero_division=0)
}

dl_metrics = {
    'accuracy': accuracy_score(y_true, dl_pred),
    'precision': precision_score(y_true, dl_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_true, dl_pred, average='weighted', zero_division=0),
    'f1': f1_score(y_true, dl_pred, average='weighted', zero_division=0)
}

print('MLP Metrics:', mlp_metrics)
print('EfficientNetB0 Metrics:', dl_metrics)

mlp_cm = confusion_matrix(y_true, mlp_pred)
dl_cm = confusion_matrix(y_true, dl_pred)

print('\nMLP Classification Report:\n')
print(classification_report(y_true, mlp_pred, target_names=class_names, digits=4))

print('\nEfficientNetB0 Classification Report:\n')
print(classification_report(y_true, dl_pred, target_names=class_names, digits=4))

## 4. Comparación visual de métricas

In [ ]:
compare_df = pd.DataFrame({
    'Métrica': ['accuracy', 'precision', 'recall', 'f1'],
    'MLP': [mlp_metrics['accuracy'], mlp_metrics['precision'], mlp_metrics['recall'], mlp_metrics['f1']],
    'EfficientNetB0': [dl_metrics['accuracy'], dl_metrics['precision'], dl_metrics['recall'], dl_metrics['f1']]
})

compare_df['Diferencia'] = compare_df['EfficientNetB0'] - compare_df['MLP']
print(compare_df)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ix = np.arange(len(compare_df))
width = 0.35
ax.bar(ix - width/2, compare_df['MLP'], width, label='MLP')
ax.bar(ix + width/2, compare_df['EfficientNetB0'], width, label='EfficientNetB0')
ax.set_xticks(ix)
ax.set_xticklabels(compare_df['Métrica'])
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Comparación de métricas')
ax.legend()

for i, v in enumerate(compare_df['MLP']):
    ax.text(i - width/2, v + 0.01, f'{v:.3f}', ha='center')
for i, v in enumerate(compare_df['EfficientNetB0']):
    ax.text(i + width/2, v + 0.01, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.show()

## 5. Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.heatmap(mlp_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('MLP')
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')

sns.heatmap(dl_cm, annot=True, fmt='d', cmap='Oranges', ax=axes[1], xticklabels=class_names, yticklabels=class_names)
axes[1].set_title('EfficientNetB0')
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()